# 電商顧客分群分析：以 RFM 模型制定行銷策略

## 專案概述

本專案針對某英國線上零售商的真實交易數據（Online Retail II，2009–2011），透過 **RFM 模型**進行顧客分群。原始數據約 105 萬筆交易，經清理後保留約 80 萬筆有效交易，目標是回答一個關鍵商業問題：

> **「行銷預算有限時，該優先把錢花在哪一群顧客身上？」**

## 核心發現

- 📊 **營收高度集中**：25.1% 的「最佳客戶」貢獻了 69.1% 的總營收，是高黏著的死忠客群（前 20% 顧客貢獻 77.2% 營收）。
- 🎯 **鎖定高 ROI 召回對象**：識別出佔 14.0% 的「重點召回」客群，他們曾有平均 4.9 次購買紀錄，是行銷投報率最高的再行銷目標。
- 💡 **策略建議**：區分「暫時性流失」（Frequency≈4.9，曾忠誠但近期沉睡）與「結構性流失」（Frequency≈1，一次性顧客），優先將預算投入前者，避免浪費資源於低忠誠顧客。

## 分析方法

1. **資料清理**：篩選有效交易（排除退貨、測試單、雜項代碼如 `POST`/`DOT`/`M`/`BANK CHARGES` 及缺失 CustomerID）。
2. **RFM 計算**：Recency（最近購買）、Frequency（購買次數）、Monetary（累積消費金額）。
3. **分群評分**：以 `pd.qcut` / SQL `NTILE(5)` 五分位數評分，切分出 6 大顧客群。
4. **策略轉化**：將數據洞察轉為可執行的行銷行動方案。

## 技術棧

`Python` · `Pandas` · `NumPy` · `Matplotlib` · `SQL (DuckDB)` · `Tableau`


---

## SQL 實作（DuckDB）

為驗證分群邏輯的正確性與可複現性，本專案以 **DuckDB 5 層 VIEW 架構**用 SQL 完整復刻 Python 分群流程：

| 層級 | VIEW | 職責 | 核心語法 |
|---|---|---|---|
| 1 | `clean_retail` | 資料清理 | `regexp_matches` · `UPPER` |
| 2 | `rfm_base` | 計算 R/F/M 原始值 | 聚合函式 (`DATE_DIFF`, `COUNT DISTINCT`) |
| 3 | `rfm_score` | 五分位評分 | `NTILE(5) OVER(...)` |
| 4 | `rfm_segment` | 分群標籤 | `CASE WHEN` 條件判斷 |
| 5 | `segment_aggregation` | 各群統計彙總 | `GROUP BY` · 營收佔比分析 |

*註：SQL 負責分群邏輯的驗證與可複現性，Pareto 累積貢獻分析（`cumsum`）與視覺化圖表則保留於 Python / Tableau 處理。*

---

## 結果驗證與 Debug 歷程（Python vs SQL）

以相同資料分別執行 Python（`pd.qcut`）與 SQL（`NTILE(5)`）分群，逐筆比對 5,878 名顧客的分群結果。

* **Debug 歷程**：初期發現不一致筆數在每次執行間浮動（一致率僅 ~92%），追查後定位到 `NTILE(5)` 的 `ORDER BY` 未指定唯一排序鍵，使同分值（大量 Frequency=1 的顧客）被隨機分組；加入 `CustomerID` 作為 tie-breaker 對齊 Python 的 `rank(method='first')` 後，**逐筆一致率提升至 99.69%**。

| 指標 | 數值 |
|---|---|
| 總顧客數 | 5,878 |
| 逐筆一致 | 5,860 |
| 不一致 | 18 |
| **逐筆一致率** | **99.69%** |
| 變異率 | 0.31% |

剩餘 0.31%（18 筆）100% 落在相鄰分群的分位邊界（例如「穩定回購」與「最佳客戶」之間），源於 `NTILE`（等量分桶）與 `qcut`（等寬分位）的演算法本質差異，非邏輯錯誤，確認 RFM 主流程跨工具結果高度一致。

## 🔗 互動式儀表板

完整互動視覺化請見 [Tableau Public 專案連結](https://your-tableau-link-here)。



In [1]:
import duckdb

# 看前 5 筆資料，確認欄位名稱
duckdb.sql("SELECT * FROM 'C:/Users/USER/Desktop/kaggle/online_retail_II.csv' LIMIT 5").df()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [14]:
duckdb.sql("""
create or replace view clean_retail as
select
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country",
    ("Quantity" * "Price") as totalprice
from 'C:/Users/USER/Desktop/kaggle/online_retail_II.csv'
where "Quantity" > 0
  and "Price" > 0
  and "Customer ID" is not null
  and not regexp_matches("Invoice", '^C')
  and not regexp_matches(UPPER("StockCode"), 'TEST');
""")

# 驗證看看
duckdb.sql("select * from clean_retail limit 5").df()


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,totalprice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [15]:
# 
duckdb.sql("""
CREATE OR REPLACE VIEW rfm_base AS      
    select
        cast("Customer ID" as bigint) as CustomerID,
        DATE_DIFF('day', max("InvoiceDate"),(select max("InvoiceDate") + interval 1 day from clean_retail)
        ) as Recency,
        COUNT(DISTINCT "Invoice") as Frequency,
        round(SUM(totalprice), 2) as Monetary
    from clean_retail
    group by "Customer ID"
""")

duckdb.sql("select * from rfm_base limit 5;").df() 


,CustomerID,Recency,Frequency,Monetary
0,14680,26,58,57412.77
1,16926,372,5,1421.35
2,13050,15,35,12344.53
3,14543,29,39,14906.77
4,12891,186,11,840.50


In [30]:
# NTILE(要分成的組數) OVER (ORDER BY 用來排序的欄位)
duckdb.sql("""
create or replace view rfm_score as
    select
        CustomerID, Recency, Frequency, Monetary,
        ntile(5) over (order by Recency desc, CustomerID)  as R_score,
        ntile(5) over (order by Frequency asc, CustomerID) as F_score,
        ntile(5) over (order by Monetary asc, CustomerID)  as M_score
    from rfm_base

""")
duckdb.sql("select * from rfm_score limit 5").df()

,CustomerID,Recency,Frequency,Monetary,R_score,F_score,M_score
0,14095,723,1,2.95,1,1,1
1,13788,506,1,3.75,1,1,1
2,16738,298,1,3.75,2,2,1
3,14792,64,1,6.20,3,1,1
4,15913,535,1,6.30,1,1,1


In [31]:
duckdb.sql("""
create or replace view rfm_segment as 
    select
        CustomerID,
        Recency, Frequency, Monetary,
        R_score, F_score, M_score,
        case
            when R_score >= 4 and F_score >= 4 then '最佳客戶'
            when R_score >= 3 and F_score >= 3 then '穩定回購'
            when R_score >= 4 and F_score <= 2 then '新客戶'
            when R_score >= 3 and F_score <= 2 then '有潛力培養'
            when R_score <= 2 and F_score >= 3 then '重點召回'
            when R_score <= 2 and F_score <= 2 then '快流失/已流失' 
            else '其他'
        end as segment
    from rfm_score
""")
duckdb.sql("select * from rfm_segment limit 5 ").df()

,CustomerID,Recency,Frequency,Monetary,R_score,F_score,M_score,segment
0,14095,723,1,2.95,1,1,1,快流失/已流失
1,13788,506,1,3.75,1,1,1,快流失/已流失
2,16738,298,1,3.75,2,2,1,快流失/已流失
3,14792,64,1,6.20,3,1,1,有潛力培養
4,15913,535,1,6.30,1,1,1,快流失/已流失


In [32]:
duckdb.sql("""
create or replace view rfm_segment_aggregation as
select
    segment,
    count(*) as 客戶數,
    round(avg(Recency), 1)  as 平均recency,
    round(avg(Frequency), 1) as 平均frequency,
    round(avg(Monetary), 2) as 平均monetary,
    round(sum(Monetary), 2) as 總營收貢獻,
    round(100.0 * sum(Monetary) / sum(sum(Monetary)) over (), 1) as 營收占比
from rfm_segment
group by segment
order by 總營收貢獻 desc
""")
duckdb.sql("select * from rfm_segment_aggregation ").df()


,segment,客戶數,平均recency,平均frequency,平均monetary,總營收貢獻,營收占比
0,最佳客戶,1473,20.6,15.7,8324.39,12261820.02,69.1
1,穩定回購,1229,78.9,5.4,2098.64,2579233.51,14.5
2,重點召回,824,369.6,4.9,1982.86,1633873.30,9.2
3,快流失/已流失,1528,459.0,1.3,438.18,669545.66,3.8
4,新客戶,439,28.4,1.5,896.63,393619.91,2.2
5,有潛力培養,385,106.3,1.4,532.76,205110.76,1.2


In [37]:
csv_path = r"C:\Users\USER\Desktop\履歷\py_rfm_result.csv"

duckdb.sql(f"""
CREATE OR REPLACE VIEW py_result AS
SELECT
    "Customer ID" AS customer_id,
    Segment       AS py_segment
FROM read_csv_auto('{csv_path.replace(chr(92), '/')}');
""")

duckdb.sql("""
CREATE OR REPLACE VIEW validation AS
SELECT
    s.CustomerID,
    s.segment      AS sql_segment,
    p.py_segment,
    CASE WHEN s.segment = p.py_segment THEN 1 ELSE 0 END AS is_match
FROM rfm_segment s
JOIN py_result   p ON s.CustomerID = p.customer_id;
""")

duckdb.sql("""
SELECT
    COUNT(*)                                        AS total,
    SUM(is_match)                                   AS matched,
    COUNT(*) - SUM(is_match)                         AS mismatched,
    ROUND(100.0 * (COUNT(*) - SUM(is_match)) / COUNT(*), 2) AS variance_pct
FROM validation;
""").show()


┌───────┬─────────┬────────────┬──────────────┐
│ total │ matched │ mismatched │ variance_pct │
│ int64 │ int128  │   int128   │    double    │
├───────┼─────────┼────────────┼──────────────┤
│  5878 │    5860 │         18 │         0.31 │
└───────┴─────────┴────────────┴──────────────┘



In [35]:
duckdb.sql("""
SELECT sql_segment, py_segment, COUNT(*) AS n
FROM validation
WHERE is_match = 0
GROUP BY sql_segment, py_segment
ORDER BY n DESC;
""").df()


,sql_segment,py_segment,n
0,穩定回購,最佳客戶,8
1,有潛力培養,新客戶,4
2,快流失/已流失,有潛力培養,4
3,重點召回,穩定回購,1
4,快流失/已流失,重點召回,1
